In [1]:
%%writefile lua/rate_limiter.lua
-- lua/rate_limiter.lua
local _M = {}

-- Parámetros del algoritmo Token Bucket (Cubo de Tokens)
local BUCKET_CAPACITY = 10  -- Ráfaga máxima permitida
local REFILL_RATE = 2       -- Tokens generados por segundo

function _M.check_rate()
    local dict = ngx.shared.limiter_storage
    local client_ip = ngx.var.remote_addr
    
    local tokens_key = client_ip .. ":tokens"
    local time_key = client_ip .. ":last_time"
    local now = ngx.now()
    
    -- Leer el estado actual de la IP en la memoria RAM compartida
    local tokens, err = dict:get(tokens_key)
    local last_time, _ = dict:get(time_key)
    
    if not tokens then
        -- Primera petición del cliente: inicializamos el cubo lleno
        tokens = BUCKET_CAPACITY
        last_time = now
    else
        -- Calcular cuántos tokens se generaron matemáticamente en base al tiempo transcurrido
        local time_passed = now - last_time
        local tokens_to_add = time_passed * REFILL_RATE
        tokens = math.min(BUCKET_CAPACITY, tokens + tokens_to_add)
    end
    
    -- Evaluación del límite
    if tokens >= 1 then
        tokens = tokens - 1
        dict:set(tokens_key, tokens)
        dict:set(time_key, now)
        return -- Retorno limpio: permite que Nginx continúe al bloque location
    else
        dict:set(time_key, now)
        -- Bloqueo inmediato en la puerta (HTTP 429)
        ngx.status = ngx.HTTP_TOO_MANY_REQUESTS
        ngx.header.content_type = "application/json; charset=utf-8"
        ngx.say([[{"error": "Too Many Requests", "message": "Freno de mano: Has excedido el límite de velocidad en el Gateway."}]])
        ngx.exit(ngx.HTTP_OK)
    end
end

-- Ejecutamos la función automáticamente al invocar el archivo mediante access_by_lua_file
_M.check_rate()

return _M

Writing lua/rate_limiter.lua


FileNotFoundError: [Errno 2] No such file or directory: 'lua/rate_limiter.lua'

In [ ]:
!pwd

In [ ]:
!docker rm -f mi-openresty-gateway

In [ ]:
!docker run -d \
  --name mi-openresty-gateway \
  -p 8080:8080 \
  -v "$(pwd)/conf/nginx.conf:/usr/local/openresty/nginx/conf/nginx.conf:ro" \
  -v "$(pwd)/lua:/usr/local/openresty/nginx/lua:ro" \
  openresty/openresty:alpine

In [ ]:
!ls

In [ ]:
Alta_optimizacion

In [ ]:
!docker run -d \
  --name mi-openresty-gateway \
  p 8080:8080 \
  v "$(pwd)/conf/nginx.conf:/usr/local/openresty/nginx/conf/nginx.conf:ro" \
  v "$(pwd)/lua:/usr/local/openresty/nginx/lua:ro" \
  openresty/openresty:alpine

In [ ]:
!openresty -p . -c conf/nginx.conf

In [2]:
!mkdir -p ~/Desktop/laboratorio_gateway/conf ~/Desktop/laboratorio_gateway/lua

In [3]:
%%writefile ~/Desktop/laboratorio_gateway/conf/nginx.conf
worker_processes 1;

events {
    worker_connections 1024;
}

http {
    lua_package_path "/usr/local/openresty/nginx/lua/?.lua;;";
    lua_shared_dict limiter_storage 10m;

    init_by_lua_block {
        require "rate_limiter"
    }

    server {
        listen 8080;
        server_name localhost;
        lua_code_cache off;

        location /api {
            access_by_lua_file "/usr/local/openresty/nginx/lua/rate_limiter.lua";
            default_type application/json;
            return 200 '{"status": "success", "message": "Petición procesada: Tráfico limpio a través del Gateway"}';
        }
    }
}

Writing /Users/admin/Desktop/laboratorio_gateway/conf/nginx.conf


In [4]:
%%writefile ~/Desktop/laboratorio_gateway/lua/rate_limiter.lua
local _M = {}

local BUCKET_CAPACITY = 10
local REFILL_RATE = 2

function _M.check_rate()
    local dict = ngx.shared.limiter_storage
    local client_ip = ngx.var.remote_addr
    
    local tokens_key = client_ip .. ":tokens"
    local time_key = client_ip .. ":last_time"
    local now = ngx.now()
    
    local tokens, err = dict:get(tokens_key)
    local last_time, _ = dict:get(time_key)
    
    if not tokens then
        tokens = BUCKET_CAPACITY
        last_time = now
    else
        local time_passed = now - last_time
        local tokens_to_add = time_passed * REFILL_RATE
        tokens = math.min(BUCKET_CAPACITY, tokens + tokens_to_add)
    end
    
    if tokens >= 1 then
        tokens = tokens - 1
        dict:set(tokens_key, tokens)
        dict:set(time_key, now)
        return
    else
        dict:set(time_key, now)
        ngx.status = ngx.HTTP_TOO_MANY_REQUESTS
        ngx.header.content_type = "application/json; charset=utf-8"
        ngx.say([[{"error": "Too Many Requests", "message": "Freno de mano: Has excedido el límite de velocidad en el Gateway."}]])
        ngx.exit(ngx.HTTP_OK)
    end
end

_M.check_rate()
return _M

Writing /Users/admin/Desktop/laboratorio_gateway/lua/rate_limiter.lua


In [5]:
!docker rm -f mi-openresty-gateway
!docker run -d \
  --name mi-openresty-gateway \
  -p 8080:8080 \
  -v ~/Desktop/laboratorio_gateway/conf/nginx.conf:/usr/local/openresty/nginx/conf/nginx.conf:ro \
  -v ~/Desktop/laboratorio_gateway/lua:/usr/local/openresty/nginx/lua:ro \
  openresty/openresty:alpine

Error: No such container: mi-openresty-gateway
a704d8e50fe8d6aa32013933e4a48fd057c6bf8bf60b6d5edd0f1fbea9029862
